# 04 — Feature Engineering
### Sales Forecasting & Business Analytics Platform — Phase 5

**Input:** `data/processed/cleaned_sales_data.csv` (from Notebook 02), informed by
every finding in Notebook 03's EDA.
**Output:** `data/processed/featured_sales_data.csv`, plus an explicit,
audited list of which columns are actually safe to feed a model in Notebook 05.


## Objectives

- Run a **feature-leakage audit** before writing a single feature — confirm
  exactly which columns are genuinely available at real forecast time, grounded
  in `test.csv`'s actual columns, not assumptions.
- Build the PRD's required date and business features, each justified by a
  specific Notebook 03 finding.
- Build optional lag/rolling features correctly — with no leakage from the
  current day's own target value.
- Persist one final feature-engineered dataset for Notebook 05.


## Theory: Why the Leakage Audit Comes First

A feature is only useful if it can actually be computed at the moment a real
prediction is needed — not just in hindsight on historical data. The single
strongest correlate with `Sales` found in Notebook 03 was `Customers` (0.82) —
but `Customers` is **not a column in `test.csv`**. A model trained to lean on it
would look excellent offline and be unusable in production, because we'd never
know next Tuesday's customer count before next Tuesday happens.

This is the single most common way a portfolio forecasting project quietly
cheats without the author noticing — validating a model on data that
accidentally contains information from the future, or information that will
never exist at prediction time. We check this explicitly, against the real
`test.csv` schema, before building anything.


In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

from src.data_loader import load_cleaned_data, load_test
from src.config import FEATURED_DATA_FILE
from src.feature_engineering import engineer_features, LEAKAGE_EXCLUDED_COLUMNS, MONTH_ABBR

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Step 0 — Feature Leakage Audit


In [2]:
cleaned = load_cleaned_data()
test = load_test()

print("Columns in cleaned train (post Notebook 02):")
print(sorted(cleaned.columns.tolist()))
print()
print("Columns in test.csv (raw, before any merge):")
print(sorted(test.columns.tolist()))

Columns in cleaned train (post Notebook 02):
['Assortment', 'CompetitionDistance', 'CompetitionDistance_was_missing', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'CompetitionOpenSince_was_missing', 'Customers', 'Date', 'DayOfWeek', 'Open', 'Promo', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Sales', 'SchoolHoliday', 'StateHoliday', 'Store', 'StoreType', 'Suspicious_Zero_Sales']

Columns in test.csv (raw, before any merge):
['Date', 'DayOfWeek', 'Id', 'Open', 'Promo', 'SchoolHoliday', 'StateHoliday', 'Store']


In [3]:
# Fair comparison requires applying the exact same store-cleaning + merge steps
# to test.csv that Notebook 02 applied to train -- comparing against a
# differently-processed test set would misrepresent which columns are truly
# unavailable. clean_store_data() adds CompetitionDistance_was_missing /
# CompetitionOpenSince_was_missing from store.csv alone, so those ARE available
# for test too, once the same pipeline step is applied.
from src.preprocessing import merge_store_metadata, clean_store_data
store = __import__("src.data_loader", fromlist=["load_store"]).load_store()
store_cleaned = clean_store_data(store)
test_merged = merge_store_metadata(test, store_cleaned)

train_only_columns = set(cleaned.columns) - set(test_merged.columns) - {"Sales"}
print("Columns in cleaned train but NOT in test.csv-after-the-identical-pipeline (excluding the target itself):")
print(sorted(train_only_columns))

Columns in cleaned train but NOT in test.csv-after-the-identical-pipeline (excluding the target itself):
['Customers', 'Suspicious_Zero_Sales']


**Audit result:** `Customers` is the only *raw* column present in training
data but absent from `test.csv` — confirming the leakage risk flagged above.
`StoreType`, `Assortment`, `CompetitionDistance`, and the rest all come from
`store.csv`, which merges onto `test.csv` the same way it merges onto `train`,
so those remain safe.

One more risk isn't visible in this column-list comparison alone: **any new
feature we engineer from `Sales` itself** (like Notebook 02's
`Suspicious_Zero_Sales` flag) is *also* unusable for real forecasting, even
though it doesn't come from `test.csv` at all — it's circular, since `Sales` is
literally what we're trying to predict.

**Decision, encoded directly in `src/feature_engineering.py`
(`LEAKAGE_EXCLUDED_COLUMNS`), not left as a verbal promise:**


In [4]:
print("Columns excluded from modeling (src/feature_engineering.py):")
for col in LEAKAGE_EXCLUDED_COLUMNS:
    print(" -", col)

Columns excluded from modeling (src/feature_engineering.py):
 - Customers
 - Suspicious_Zero_Sales


This list is imported directly by Notebook 05 later — not just written down
here and hoped to be remembered.


## Step 1 — Date Features
**PRD requirement:** Year, Month, Day, Week, Quarter, Weekday, Weekend.

**Why:** `Date` alone isn't usable by most models directly — decomposing it into
calendar parts lets a model learn patterns like "December is stronger" (Notebook
03, chart #2) or "Q3 is the softest quarter" (chart #4) as structured, reusable
signal instead of memorizing every individual date.

**Business meaning:** these are the calendar building blocks any retail
forecaster needs — a model can't learn "December is busy" from a raw date
string, but it can from a `Month == 12` feature seen repeatedly across 3 years.

**Advantages:** cheap to compute, zero leakage risk (available for any future
date), and directly grounded in confirmed EDA patterns rather than guessed.

**Possible drawback:** `Year` risks the model learning a spurious linear "sales
increase with year" pattern if extrapolated far beyond the training window —
worth watching in Notebook 06's evaluation, especially since Notebook 03 found
no strong genuine multi-year trend to begin with.


In [5]:
featured = engineer_features(cleaned)

date_feature_cols = ["Date", "Year", "Month", "Day", "WeekOfYear", "Quarter", "DayOfWeek", "IsWeekend"]
featured[date_feature_cols].head()

,Date,Year,Month,Day,WeekOfYear,Quarter,DayOfWeek,IsWeekend
1016095,2013-01-01,2013,1,1,1,1,2,False
1014980,2013-01-02,2013,1,2,1,1,3,False
1013865,2013-01-03,2013,1,3,1,1,4,False
1012750,2013-01-04,2013,1,4,1,1,5,False
1011635,2013-01-05,2013,1,5,1,1,6,True


**Expected output:** each row now carries its own Year/Month/Day/Week/Quarter
decomposition alongside the existing `DayOfWeek`, plus the new `IsWeekend` flag.


## Step 2 — Holiday Flag
**PRD requirement:** Holiday Flag.

**Why:** `StateHoliday` already carries detail (`'a'`/`'b'`/`'c'`), but a simple
binary "is this a holiday at all" signal is easier for a model (and a dashboard
viewer) to use directly, and follows straight from Notebook 03's Holiday Impact
chart.

**Business meaning:** answers "should today be treated as calendar-atypical" in
one flag.

**Advantages:** simple, interpretable, zero leakage risk.

**Possible drawback:** collapses three genuinely different holiday types (public
holiday, Easter, Christmas — each showed a different average in Notebook 03)
into one flag; `StateHoliday` itself remains in the dataset for models that can
use the more granular version.


In [6]:
featured["IsHoliday"].value_counts()

IsHoliday
False    986159
True      31050
Name: count, dtype: int64

## Step 3 — Recurring Promotion Activity Flag (`IsPromo2Active`)
**PRD requirement:** Promotion Flag (this goes beyond the raw `Promo`/`Promo2`
columns already in the data).

**Why:** Notebook 03's Promo2 chart found *lower* average sales for
Promo2-enrolled stores — but flagged that as likely a selection effect, since
raw `Promo2` only says a store is *ever* enrolled, not *when* its promotion is
actually running. `PromoInterval` (e.g. `"Feb,May,Aug,Nov"`) specifies the exact
months. This feature checks the row's own month against that list.

**A real gotcha caught while building this:** the dataset spells September as
**`"Sept"`** (4 letters) in `PromoInterval`, not the standard 3-letter `"Sep"`
that Python's `datetime.strftime("%b")` produces. Using the standard formatter
would have silently zeroed out this feature for every September row across
every enrolled store. Verified directly before writing the function — see the
comparison below.


In [7]:
from datetime import datetime
print("Python's standard abbreviation:", datetime(2020, 9, 1).strftime("%b"))
print("Dataset's actual spelling:     ", "Sept")
print("Custom MONTH_ABBR map used instead:", MONTH_ABBR[9])

Python's standard abbreviation: Sep
Dataset's actual spelling:      Sept
Custom MONTH_ABBR map used instead: Sept


In [8]:
promo2_check = featured[featured["Promo2"] == 1][["Store","Month","PromoInterval","IsPromo2Active"]] \
    .drop_duplicates(subset=["Store","Month"]).sort_values(["Store","Month"]).head(12)
promo2_check

,Store,Month,PromoInterval,IsPromo2Active
1016096,2,1,"Jan,Apr,Jul,Oct",True
981531,2,2,"Jan,Apr,Jul,Oct",False
950311,2,3,"Jan,Apr,Jul,Oct",False
915746,2,4,"Jan,Apr,Jul,Oct",True
882296,2,5,"Jan,Apr,Jul,Oct",False
847731,2,6,"Jan,Apr,Jul,Oct",False
814281,2,7,"Jan,Apr,Jul,Oct",True
779716,2,8,"Jan,Apr,Jul,Oct",False
745151,2,9,"Jan,Apr,Jul,Oct",False
711701,2,10,"Jan,Apr,Jul,Oct",True


**Business meaning:** a much more precise "is a recurring discount actually
live right now" signal than the store-level enrollment flag alone.

**Advantages:** gives the model a chance to find the *true* Promo2 relationship
(Notebook 03 explicitly could not conclude one from the coarse flag alone);
directly demonstrates the parsing gotcha is handled correctly (`IsPromo2Active`
is `True` only in the exact 4 months per store's own interval, `False`
elsewhere, and always `False` when `Promo2 == 0`).

**Possible drawback:** doesn't capture campaign *intensity* or exact start/end
dates within the active months — it's month-level granularity, not day-level.


In [9]:
print("IsPromo2Active True count:", featured["IsPromo2Active"].sum())
print("Confirms False whenever Promo2==0:", featured.loc[featured['Promo2']==0, 'IsPromo2Active'].sum() == 0)

IsPromo2Active True count: 174792
Confirms False whenever Promo2==0: True


## Step 4 — Lag and Rolling Features (Optional, PRD)
**PRD requirement:** Previous Day Sales, Previous Week Sales, Rolling Mean.

**Why:** recent sales history is often one of the strongest predictors of
near-future sales in retail forecasting — yesterday being busy is real
information about today.

**Leakage-safety requirement:** pandas' `.rolling()` includes the *current* row
by default. Using it directly on `Sales` would leak each row's own target value
into its own feature — a subtle bug that can make a model look artificially
excellent offline while being useless in production. Every feature below is
built on `Sales.shift(1)` first (yesterday and earlier only), verified with a
real store's chronological history below.


In [10]:
lag_cols = ["Store","Date","Sales","PrevDaySales","PrevWeekSales","RollingMean7","RollingMean30"]
featured[featured["Store"] == 1].sort_values("Date")[lag_cols].head(10)

,Store,Date,Sales,PrevDaySales,PrevWeekSales,RollingMean7,RollingMean30
1016095,1,2013-01-01,0,NaN,NaN,NaN,NaN
1014980,1,2013-01-02,5530,0.0,NaN,0.000000,0.000000
1013865,1,2013-01-03,4327,5530.0,NaN,2765.000000,2765.000000
1012750,1,2013-01-04,4486,4327.0,NaN,3285.666667,3285.666667
1011635,1,2013-01-05,4997,4486.0,NaN,3585.750000,3585.750000
1010520,1,2013-01-06,0,4997.0,NaN,3868.000000,3868.000000
1009405,1,2013-01-07,7176,0.0,NaN,3223.333333,3223.333333
1008290,1,2013-01-08,5580,7176.0,0.0,3788.000000,3788.000000
1007175,1,2013-01-09,5471,5580.0,5530.0,4585.142857,4012.000000
1006060,1,2013-01-10,4892,5471.0,4327.0,4576.714286,4174.111111


**Verification of the sample above:** Store 1's `PrevDaySales` for
2013-01-02 correctly equals 2013-01-01's actual `Sales` (not its own day's
value); `RollingMean7` for the second row equals just the first day's sales
(only one prior observation exists yet) and grows correctly from there. This
confirms no same-day leakage.

**Business meaning:** captures short-term momentum (yesterday, last week) and
medium-term baseline level (7-day, 30-day averages) per store.

**Advantages:** typically strong predictive signal in retail time series;
grounded in verified, leakage-safe computation.

**Possible drawbacks:**
1. Each store's earliest rows have no history yet, producing `NaN` — a real,
   visible gap (not silently filled here), which Notebook 05 must explicitly
   decide how to handle.
2. These features use raw `Sales` history *including* closed-store days
   (`Sales == 0` when `Open == 0`) — a "previous day" that was a closed day
   will show `0`, which is truthful but means the model needs `Open` alongside
   these features to interpret a `0` correctly (closed vs genuinely no
   demand).


In [11]:
nan_counts = featured[["PrevDaySales","PrevWeekSales","RollingMean7","RollingMean30"]].isnull().sum()
print(nan_counts)
print()
print(f"PrevWeekSales NaN count ({nan_counts['PrevWeekSales']:,}) should equal 7 x number of stores ({7 * featured['Store'].nunique():,})")

PrevDaySales     1115
PrevWeekSales    7805
RollingMean7     1115
RollingMean30    1115
dtype: int64

PrevWeekSales NaN count (7,805) should equal 7 x number of stores (7,805)


## Step 5 — Final Feature Set & Persistence


In [12]:
all_columns = featured.columns.tolist()
safe_columns = [c for c in all_columns if c not in LEAKAGE_EXCLUDED_COLUMNS]

print(f"Total columns: {len(all_columns)}")
print(f"Safe for modeling: {len(safe_columns)}")
print(f"Excluded (leakage risk): {LEAKAGE_EXCLUDED_COLUMNS}")

Total columns: 33
Safe for modeling: 31
Excluded (leakage risk): ['Customers', 'Suspicious_Zero_Sales']


In [13]:
FEATURED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
featured.to_csv(FEATURED_DATA_FILE, index=False)

import os
size_mb = os.path.getsize(FEATURED_DATA_FILE) / (1024 * 1024)
print(f"Saved to: {FEATURED_DATA_FILE}")
print(f"File size: {size_mb:.1f} MB")

Saved to: /home/claude/Sales-Forecasting/data/processed/featured_sales_data.csv
File size: 162.9 MB


**Note:** the saved file keeps `Customers` and `Suspicious_Zero_Sales` in
place (useful for further EDA or debugging) — they are excluded at the
*modeling* stage via `LEAKAGE_EXCLUDED_COLUMNS`, not deleted from the dataset
itself. Keeping the audit as an explicit, importable list rather than deleting
columns means the decision is enforced in code, not just remembered.


## Feature Engineering Summary Report

| Feature | Type | Business Meaning | Advantage | Drawback |
|---|---|---|---|---|
| `Year`, `Month`, `Day`, `WeekOfYear`, `Quarter` | Date | Calendar decomposition | Cheap, zero leakage, grounded in EDA | `Year` risks spurious trend extrapolation |
| `IsWeekend` | Business | Sat/Sun flag | Lets model learn true (non-obvious) weekend effect | None significant |
| `IsHoliday` | Business | Binary holiday summary | Simple, interpretable | Collapses 3 distinct holiday types into 1 |
| `IsPromo2Active` | Business | Recurring promo actually running this month | More precise than raw enrollment flag; follows up ambiguous EDA finding | Month-level, not day-level granularity |
| `PrevDaySales`, `PrevWeekSales` | Lag (optional) | Recent sales momentum | Strong typical retail signal | NaN at each store's history start |
| `RollingMean7`, `RollingMean30` | Rolling (optional) | Short/medium-term baseline level | Smooths noise, leakage-safe (shifted) | Same NaN issue; blends open/closed days |
| `Customers` | **Excluded from modeling** | Not available in test.csv | — | Would silently break real forecasting if used |
| `Suspicious_Zero_Sales` | **Excluded from modeling** | Derived from Sales itself | — | Circular — cannot exist for unseen rows |


## Business Observations

- The leakage audit is arguably the most important output of this notebook —
  it's easy to build an offline-impressive model on `Customers` and never
  notice it's unusable in production until deployment.
- `IsPromo2Active` exists specifically to try to resolve the ambiguous Promo2
  finding from Notebook 03 with more precise information — Notebook 06's
  feature importance analysis will show whether it actually helps.
- The lag/rolling features assume per-store daily continuity; if this pipeline
  is ever extended to new stores with no sales history, those stores would
  start with entirely `NaN` lag features — worth remembering for any future
  "new store" scenario, though out of scope for the current PRD.


## Next Steps

Notebook 05 (`05_model_training.ipynb`) will import `LEAKAGE_EXCLUDED_COLUMNS`
directly from `src/feature_engineering.py`, build the chronological
train/validation split decided in the architecture review, and train Linear
Regression and Random Forest models (with XGBoost/LightGBM as time permits) —
using only the audited, leakage-safe feature set built here.
